# Introduction to Database AI Agents

A **database AI agent** connects natural language to **structured data** in tables — orders, customers, inventory, metrics — through a controlled query loop instead of guessing from prose or parametric memory.

```
User: "How many shipped orders over $50 last week?"
         │
         ▼
   Database AI Agent
         │
    ┌────┴────┐
    ▼         ▼
 schema     generate
 context    SELECT …
    │         │
    └────┬────┘
         ▼
   validate (read-only, allowlist)
         ▼
   execute → rows → grounded answer + SQL citation
```

This notebook builds **ShopCo Analytics & Support DB Agent** on an in-memory **SQLite** database:

1. Model a realistic **e-commerce schema** (customers, orders, products)
2. Expose **schema metadata** the agent needs for grounding
3. Run a **safe SQL execution layer** (SELECT-only by default)
4. Implement an **offline NL→SQL planner** and full agent loop
5. Compare **ungrounded** vs **database-grounded** answers
6. Preview **guardrails** covered in depth later in this module

Self-contained plain Python + `sqlite3`. No CLI or external database server required.

In [ ]:
"""
ShopCo Database AI Agent — introduction lab.
"""

import json
import os
import re
import sqlite3
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


print("ShopCo database agent lab ready")

---

## 1. What Is a Database AI Agent?

| Component | Role |
|-----------|------|
| **Schema context** | Table/column names, types, relationships, sample values |
| **Query generator** | NL → SQL (LLM or rules) |
| **Validator** | Block destructive SQL, enforce read-only, row limits |
| **Executor** | Run against DB connection with timeouts |
| **Answer composer** | Turn rows into natural language + SQL citation |

**Not** a database AI agent: stuffing table CSVs into the prompt and hoping the model memorizes rows. **Is** a database AI agent: executing verified queries at answer time.

---

## 2. ShopCo SQLite Schema

We use `:memory:` SQLite — ephemeral, zero setup, same SQL patterns as Postgres/BigQuery agents.

In [ ]:
def create_shopco_db() -> sqlite3.Connection:
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            email TEXT NOT NULL UNIQUE,
            name TEXT NOT NULL,
            tier TEXT DEFAULT 'standard'
        );
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            sku TEXT NOT NULL UNIQUE,
            name TEXT NOT NULL,
            price_usd REAL NOT NULL
        );
        CREATE TABLE orders (
            order_id TEXT PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
            status TEXT NOT NULL,
            total_usd REAL NOT NULL,
            tracking TEXT,
            created_at TEXT NOT NULL,
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        );
        CREATE TABLE order_items (
            item_id INTEGER PRIMARY KEY,
            order_id TEXT NOT NULL REFERENCES orders(order_id),
            product_id INTEGER NOT NULL REFERENCES products(product_id),
            quantity INTEGER NOT NULL,
            FOREIGN KEY (order_id) REFERENCES orders(order_id),
            FOREIGN KEY (product_id) REFERENCES products(product_id)
        );
    """)
    conn.executemany(
        "INSERT INTO customers VALUES (?, ?, ?, ?)",
        [(1, "alice@shopco.com", "Alice Nguyen", "gold"),
         (2, "bob@shopco.com", "Bob Smith", "standard"),
         (3, "carol@shopco.com", "Carol Lee", "standard")],
    )
    conn.executemany(
        "INSERT INTO products VALUES (?, ?, ?, ?)",
        [(1, "MOUSE-01", "Wireless Mouse", 29.99),
         (2, "HUB-02", "USB-C Hub", 49.99),
         (3, "LAMP-03", "Desk Lamp", 42.00)],
    )
    conn.executemany(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?)",
        [
            ("ORD-1001", 1, "shipped", 89.50, "1Z999AA10123456784", "2026-07-01"),
            ("ORD-1002", 2, "processing", 42.00, None, "2026-07-08"),
            ("ORD-1003", 3, "delivered", 120.00, "1Z999AA10987654321", "2026-06-15"),
        ],
    )
    conn.executemany(
        "INSERT INTO order_items VALUES (?, ?, ?, ?)",
        [(1, "ORD-1001", 1, 1), (2, "ORD-1001", 2, 1), (3, "ORD-1002", 3, 1), (4, "ORD-1003", 2, 2)],
    )
    conn.commit()
    return conn


DB = create_shopco_db()
tables = DB.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
print("Tables:", [t["name"] for t in tables])

---

## 3. Schema Registry — Agent-Facing Metadata

Models need more than raw `CREATE TABLE` — include **column descriptions**, **FK hints**, and **example values**.

In [ ]:
SCHEMA_REGISTRY: dict[str, dict[str, Any]] = {
    "customers": {
        "description": "ShopCo customer accounts",
        "columns": {
            "customer_id": "PK integer",
            "email": "unique customer email",
            "name": "display name",
            "tier": "loyalty tier: standard | gold",
        },
    },
    "products": {
        "description": "Product catalog",
        "columns": {
            "product_id": "PK integer",
            "sku": "stock keeping unit code",
            "name": "product display name",
            "price_usd": "unit price in USD",
        },
    },
    "orders": {
        "description": "Customer orders — use order_id like ORD-1001",
        "columns": {
            "order_id": "PK text ORD-XXXX",
            "customer_id": "FK → customers.customer_id",
            "status": "processing | shipped | delivered | cancelled",
            "total_usd": "order total USD",
            "tracking": "carrier tracking number or NULL",
            "created_at": "ISO date string",
        },
    },
    "order_items": {
        "description": "Line items per order",
        "columns": {
            "item_id": "PK integer",
            "order_id": "FK → orders.order_id",
            "product_id": "FK → products.product_id",
            "quantity": "units ordered",
        },
    },
}


def schema_prompt_block(registry: dict[str, dict[str, Any]]) -> str:
    lines = ["# ShopCo Database Schema"]
    for table, meta in registry.items():
        lines.append(f"\n## {table} — {meta['description']}")
        for col, desc in meta["columns"].items():
            lines.append(f"  - {col}: {desc}")
    return "\n".join(lines)


print(schema_prompt_block(SCHEMA_REGISTRY)[:400], "\n...")

---

## 4. Safe SQL Execution Layer

Production agents **never** pass raw LLM strings directly to the database. Validate first:

- **SELECT only** (for analytics/support agents)
- Reject `DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`
- Enforce `LIMIT` if missing
- Log every query for audit

In [ ]:
FORBIDDEN_SQL = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|REPLACE|TRUNCATE|ATTACH|DETACH|PRAGMA)\b",
    re.IGNORECASE,
)

SQL_AUDIT_LOG: list[dict[str, Any]] = []


@dataclass
class QueryResult:
    sql: str
    rows: list[dict[str, Any]]
    row_count: int
    error: str | None = None

    @property
    def ok(self) -> bool:
        return self.error is None


class SQLValidationError(Exception):
    pass


def validate_sql(sql: str, *, allow_writes: bool = False) -> str:
    cleaned = sql.strip().rstrip(";")
    if not cleaned:
        raise SQLValidationError("empty query")
    if not allow_writes and FORBIDDEN_SQL.search(cleaned):
        raise SQLValidationError("only SELECT queries allowed")
    if not re.match(r"^SELECT\b", cleaned, re.IGNORECASE):
        raise SQLValidationError("query must start with SELECT")
    if "limit" not in cleaned.lower():
        cleaned += " LIMIT 100"
    return cleaned


def execute_sql(conn: sqlite3.Connection, sql: str, *, agent_id: str = "db_agent") -> QueryResult:
    try:
        safe_sql = validate_sql(sql)
        cur = conn.execute(safe_sql)
        rows = [dict(r) for r in cur.fetchall()]
        SQL_AUDIT_LOG.append({"ts": utc_now(), "agent": agent_id, "sql": safe_sql, "rows": len(rows), "ok": True})
        return QueryResult(sql=safe_sql, rows=rows, row_count=len(rows))
    except (SQLValidationError, sqlite3.Error) as exc:
        SQL_AUDIT_LOG.append({"ts": utc_now(), "agent": agent_id, "sql": sql, "ok": False, "error": str(exc)})
        return QueryResult(sql=sql, rows=[], row_count=0, error=str(exc))


demo = execute_sql(DB, "SELECT order_id, status, total_usd FROM orders WHERE status = 'shipped'")
print(pretty(demo))
blocked = execute_sql(DB, "DELETE FROM orders")
print("Blocked:", blocked.error)

---

## 5. RAG vs Database Agent — When to Use Which

| Question type | Right approach | Why |
|---------------|----------------|-----|
| "What is the return policy?" | **RAG** over policy docs | Prose in documents |
| "Status of ORD-1001?" | **SQL** on `orders` | Exact row, changes hourly |
| "How many gold-tier customers?" | **SQL** `COUNT` + `JOIN` | Aggregation over tables |
| "Explain warranty in plain English" | **RAG** | Narrative policy text |
| "Total revenue from Desk Lamps" | **SQL** join `order_items` → `products` | Structured math |

Database agents excel at **fresh, exact, tabular** facts. RAG excels at **unstructured prose**.

---

## 6. Offline NL → SQL Planner

Rule-based planner mimics an LLM generating SQL — same agent loop, no API cost.

In [ ]:
def offline_nl_to_sql(question: str) -> str | None:
    q = question.lower()
    m = re.search(r"ORD-[0-9]{4}", question.upper())

    if m:
        return f"SELECT order_id, status, total_usd, tracking, created_at FROM orders WHERE order_id = '{m.group(0)}'"

    if "how many" in q and "order" in q:
        if "shipped" in q:
            return "SELECT COUNT(*) AS order_count FROM orders WHERE status = 'shipped'"
        return "SELECT COUNT(*) AS order_count FROM orders"

    if "gold" in q and "customer" in q:
        return "SELECT customer_id, email, name FROM customers WHERE tier = 'gold'"

    if "desk lamp" in q or "lamp" in q:
        return """
            SELECT p.name, SUM(oi.quantity) AS units_sold, SUM(oi.quantity * p.price_usd) AS revenue_usd
            FROM order_items oi
            JOIN products p ON oi.product_id = p.product_id
            WHERE p.name LIKE '%Lamp%'
            GROUP BY p.name
        """.strip()

    if "tracking" in q and "shipped" in q:
        return "SELECT order_id, tracking FROM orders WHERE status = 'shipped' AND tracking IS NOT NULL"

    if "highest" in q or "largest order" in q:
        return "SELECT order_id, total_usd FROM orders ORDER BY total_usd DESC LIMIT 1"

    return None


for q in ["ORD-1001 status", "How many shipped orders?", "Revenue from Desk Lamp"]:
    sql = offline_nl_to_sql(q)
    print(f"Q: {q}")
    print(f"  SQL: {sql}\n" if sql else "  SQL: (no match)\n")

---

## 7. Database Agent State and Loop

The agent loop: **classify → plan SQL → validate → execute → compose answer**.

In [ ]:
class QueryIntent(str, Enum):
    ORDER_LOOKUP = "order_lookup"
    AGGREGATE = "aggregate"
    CUSTOMER = "customer"
    PRODUCT = "product"
    UNKNOWN = "unknown"


@dataclass
class DatabaseAgentState:
    run_id: str
    question: str
    intent: QueryIntent = QueryIntent.UNKNOWN
    planned_sql: str = ""
    query_result: QueryResult | None = None
    answer: str = ""
    grounded: bool = False
    trace: list[str] = field(default_factory=list)

    def log(self, msg: str) -> None:
        self.trace.append(msg)


def classify_db_intent(question: str) -> QueryIntent:
    q = question.lower()
    if re.search(r"ORD-[0-9]{4}", question.upper()) or "order" in q:
        if any(w in q for w in ("how many", "count", "total revenue", "highest")):
            return QueryIntent.AGGREGATE
        return QueryIntent.ORDER_LOOKUP
    if "customer" in q:
        return QueryIntent.CUSTOMER
    if any(w in q for w in ("product", "lamp", "mouse", "sku")):
        return QueryIntent.PRODUCT
    return QueryIntent.UNKNOWN


def compose_db_answer(state: DatabaseAgentState) -> DatabaseAgentState:
    qr = state.query_result
    if not qr or not qr.ok:
        state.answer = f"I could not query the database: {qr.error if qr else 'no query'}"
        state.grounded = False
        state.log("compose:error")
        return state
    if not qr.rows:
        state.answer = "No matching rows in the ShopCo database."
        state.grounded = True
        state.log("compose:empty")
        return state
    state.grounded = True
    if len(qr.rows) == 1 and "order_id" in qr.rows[0]:
        row = qr.rows[0]
        state.answer = (
            f"Order {row.get('order_id')}: status={row.get('status')}, "
            f"total=${row.get('total_usd')}, tracking={row.get('tracking') or 'pending'}."
        )
    else:
        state.answer = f"Query returned {qr.row_count} row(s): {pretty(qr.rows[:3])}"
    state.answer += f"\n\n[SQL] {qr.sql}"
    state.log("compose:ok")
    return state

---

## 8. `DatabaseAgent` — Full Orchestrator

In [ ]:
class DatabaseAgent:
    def __init__(self, conn: sqlite3.Connection, schema_registry: dict[str, dict[str, Any]]) -> None:
        self.conn = conn
        self.schema_registry = schema_registry

    def run(self, question: str) -> DatabaseAgentState:
        state = DatabaseAgentState(run_id=str(uuid.uuid4())[:8], question=question)
        state.intent = classify_db_intent(question)
        state.log(f"intent:{state.intent.value}")

        sql = offline_nl_to_sql(question)
        if not sql:
            state.answer = "I need a clearer database question (order ID, counts, products, customers)."
            state.log("plan:no_sql")
            return state

        state.planned_sql = sql
        state.log("plan:sql")
        state.query_result = execute_sql(self.conn, sql)
        state.log(f"execute:ok={state.query_result.ok}")
        return compose_db_answer(state)

    def schema_context(self) -> str:
        return schema_prompt_block(self.schema_registry)


db_agent = DatabaseAgent(DB, SCHEMA_REGISTRY)

for q in ["What is status of ORD-1001?", "How many shipped orders?", "Revenue from Desk Lamp"]:
    out = db_agent.run(q)
    print(f"\nQ: {q}")
    print(f"  grounded={out.grounded} | trace={out.trace}")
    print(f"  {out.answer[:100]}...")

---

## 9. Multi-Table Join Example

Real database agents routinely **JOIN** tables — schema registry must document FK paths.

In [ ]:
join_sql = """
SELECT c.name, c.email, o.order_id, o.status, p.name AS product, oi.quantity
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.order_id = 'ORD-1001'
"""
join_result = execute_sql(DB, join_sql)
print(f"Join rows: {join_result.row_count}")
for row in join_result.rows:
    print(f"  {row['name']} bought {row['quantity']}x {row['product']} ({row['status']})")

---

## 10. Ungrounded vs Database-Grounded

Compare parametric guesses with SQL-backed answers.

In [ ]:
def ungrounded_guess(question: str) -> str:
    """Simulates LLM without DB access — may invent counts."""
    if "how many" in question.lower():
        return "There are approximately 50 shipped orders."  # wrong
    if "ORD-1001" in question.upper():
        return "ORD-1001 is probably still processing."  # wrong
    return "I'm not sure about your order data."


q = "How many shipped orders?"
grounded = db_agent.run(q)
print("Ungrounded:", ungrounded_guess(q))
print("Grounded:", grounded.answer.split("\n")[0])
print("Actual count:", grounded.query_result.rows[0] if grounded.query_result else None)

---

## 11. Agent Tools Interface

In multi-agent systems, the database layer exposes tools the planner can call.

In [ ]:
def list_tables_tool() -> dict[str, Any]:
    return {"tables": list(SCHEMA_REGISTRY.keys())}


def describe_schema_tool(table: str) -> dict[str, Any]:
    if table not in SCHEMA_REGISTRY:
        return {"error": f"unknown table {table}"}
    return SCHEMA_REGISTRY[table]


def run_read_query_tool(sql: str) -> dict[str, Any]:
    result = execute_sql(DB, sql)
    return {"sql": result.sql, "rows": result.rows, "error": result.error}


DB_TOOLS: dict[str, Callable[..., dict[str, Any]]] = {
    "list_tables": list_tables_tool,
    "describe_schema": describe_schema_tool,
    "run_read_query": run_read_query_tool,
}

print("DB tools:", list(DB_TOOLS.keys()))
print(pretty(DB_TOOLS["describe_schema"]("orders")))

---

## 12. ReAct-Style Database Agent Loop

Advanced agents may **inspect schema** before generating SQL — especially on unfamiliar warehouses.

In [ ]:
def react_db_agent(question: str, max_steps: int = 4) -> dict[str, Any]:
    trace: list[str] = []
    trace.append(f"step1:list_tables → {DB_TOOLS['list_tables']()}")

    if "lamp" in question.lower() or "product" in question.lower():
        trace.append(f"step2:describe products → {DB_TOOLS['describe_schema']('products')['description']}")
        sql = offline_nl_to_sql(question)
    elif re.search(r"ORD-[0-9]{4}", question.upper()):
        trace.append("step2:describe orders")
        sql = offline_nl_to_sql(question)
    else:
        sql = offline_nl_to_sql(question)

    if not sql:
        return {"answer": "Could not plan SQL", "trace": trace}

    result = DB_TOOLS["run_read_query"](sql)
    trace.append(f"step3:execute → {result['sql'][:60]}... rows={len(result.get('rows', []))}")
    answer = f"Found {len(result.get('rows', []))} rows. Top: {result.get('rows', [])[:2]}"
    return {"answer": answer, "sql": result.get("sql"), "trace": trace}


react_out = react_db_agent("Revenue from Desk Lamp products")
print(react_out["answer"])
print("Trace:", react_out["trace"])

---

## 13. SQL Audit Log

In [ ]:
print(f"Audit entries: {len(SQL_AUDIT_LOG)}")
for entry in SQL_AUDIT_LOG[-5:]:
    status = "OK" if entry.get("ok") else "ERR"
    print(f"  [{status}] {entry.get('sql', '')[:70]}...")

---

## 14. Risks and Guardrails Preview

| Risk | Mitigation (covered later in module) |
|------|--------------------------------------|
| SQL injection | Parameterized queries; validate identifiers |
| Destructive writes | SELECT-only default; role-based allowlist |
| Schema hallucination | Schema registry + `describe_schema` tool |
| Unbounded scans | Mandatory `LIMIT`; query timeouts |
| PII leakage | Column masks; row-level security |
| Wrong join path | FK documentation; few-shot examples |

---

## 15. Architecture in a Mixed Data Agent

```
                    ShopCo Support Coordinator
                              │
              ┌───────────────┼───────────────┐
              ▼               ▼               ▼
         Policy RAG     Database Agent    External APIs
         (documents)     (this notebook)   (tracking)
              │               │               │
         FAISS / BM25      SQLite/PG        tools/MCP
```

Database agents handle **tabular truth**. They do not replace RAG for policy prose.

---

## 16. Scenario Suite

In [ ]:
SUITE = [
    "Status of ORD-1002",
    "How many shipped orders do we have?",
    "Which customers are gold tier?",
    "What is the highest value order?",
    "Tell me a joke about SQL",
]

for q in SUITE:
    r = db_agent.run(q)
    print(f"\nQ: {q}")
    print(f"  intent={r.intent.value} grounded={r.grounded}")
    print(f"  {r.answer.split(chr(10))[0][:90]}")

---

## 17. Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| RAG over table CSV exports | Stale/wrong aggregates | Live SQL agent |
| No SQL validation | DROP TABLE from prompt injection | SELECT-only gate |
| Schema dump without descriptions | Wrong column names in SQL | Rich schema registry |
| No query audit | Compliance failure | Log SQL + row counts |
| Write access by default | Data corruption | Read-only role |
| Single-shot SQL | Wrong table guess | ReAct: list_tables → describe → query |

---

## 18. Optional Live LLM SQL Generator

Replace `offline_nl_to_sql` with an LLM that sees `schema_context()` and returns SELECT only.

In [ ]:
def live_nl_to_sql(question: str, schema: str) -> str | None:
    if not USE_LIVE_LLM:
        return offline_nl_to_sql(question)
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    prompt = (
        f"{schema}\n\nWrite a single SQLite SELECT query for: {question}\n"
        "Rules: SELECT only, include LIMIT, no comments."
    )
    return llm.invoke(prompt).content.strip().strip("`").replace("sql", "").strip()


sql = live_nl_to_sql("count shipped orders", db_agent.schema_context())
print("Planned SQL:", sql)
if sql:
    print(execute_sql(DB, sql))

---

## 19. Quiz

1. What are the five components of a database AI agent loop?
2. Why should support agents use SQL for order status instead of RAG?
3. What SQL statements should a read-only support agent block?
4. What belongs in a schema registry beyond column types?
5. How does a ReAct-style agent differ from single-shot text-to-SQL?

---

## 20. Summary

A **database AI agent** grounds answers in **live query results** from structured tables. The ShopCo lab demonstrated:

- In-memory **SQLite** schema with customers, orders, products, line items
- **Schema registry** for LLM-facing metadata
- **Safe executor** — SELECT-only, auto-`LIMIT`, audit log
- **Offline NL→SQL planner** and full `DatabaseAgent` loop
- **Tool interface** (`list_tables`, `describe_schema`, `run_read_query`)
- **ReAct** pattern for schema inspection before querying

Next steps in this module: **schema grounding** for complex warehouses, **SQL guardrails**, and **combining RAG + SQL** in one coordinator workflow.